In [1]:
import pandas as pd
import  numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("cleaned_data_raw.csv")
os.makedirs("eda_charts", exist_ok=True)

In [3]:
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi":120, "figure.facecolor":"white",
                    "axes.spines.top":False, "axes.spines.right":False})

def save(name):
    plt.tight_layout()
    plt.savefig(f"eda_charts/{name}.png", bbox_inches="tight")
    plt.close()
    print(f" {name}.png")

print("Generating 20 EDA charts...")

Generating 20 EDA charts...


In [23]:
# Price Distribution
fig, ax = plt.subplots(figsize=(9,5))
sns.histplot(df['Price_in_Lakhs'], bins=40, kde=True, color="#4C72B0", ax=ax)
ax.axvline(df['Price_in_Lakhs'].median(), color='red', linestyle='--',
           label=f"Median ₹{df['Price_in_Lakhs'].median():.1f}L")
ax.set_title("1 — Distribution of Property Prices", fontweight="bold")
ax.set_xlabel("Price (Lakhs ₹)"); ax.legend()
save("chart01_price_distribution")

 chart01_price_distribution.png


In [4]:
# Size Distribution
fig, ax = plt.subplots(figsize=(9,5))
sns.histplot(df['Size_in_SqFt'], bins=40, kde=True, color="#55A868", ax=ax)
ax.axvline(df['Size_in_SqFt'].median(), color='red', linestyle='--',
           label=f"Median {df['Size_in_SqFt'].median():.0f} sqft")
ax.set_title("2 — Distribution of Property Sizes", fontweight="bold")
ax.set_xlabel("Size (sq ft)"); ax.legend()
save("chart02_size_distribution")

 chart02_size_distribution.png


In [5]:
# Price per SqFt by Property Type 
fig, ax = plt.subplots(figsize=(9,5))
order = df.groupby('Property_Type')['Price_per_SqFt'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Property_Type', y='Price_per_SqFt', order=order,
            palette="Set2", ax=ax, flierprops=dict(marker='o', markersize=3, alpha=0.4))
ax.set_title("3 — Price per SqFt by Property Type", fontweight="bold")
save("chart03_price_per_sqft_by_type")



 chart03_price_per_sqft_by_type.png


In [6]:
# Size vs Price Scatter
fig, ax = plt.subplots(figsize=(9,5))
sc = ax.scatter(df['Size_in_SqFt'], df['Price_in_Lakhs'],
                c=df['BHK'], cmap='viridis', alpha=0.5, s=20)
plt.colorbar(sc, ax=ax, label="BHK")
ax.set_title("4 — Property Size vs Price (color=BHK)", fontweight="bold")
ax.set_xlabel("Size (sqft)"); ax.set_ylabel("Price (Lakhs ₹)")
save("chart04_size_vs_price")

 chart04_size_vs_price.png


In [7]:
# Outlier detection
fig, axes = plt.subplots(1,2,figsize=(11,5))
for ax, col, color in zip(axes, ['Price_in_Lakhs','Size_in_SqFt'], ['#4C72B0','#55A868']):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(f"Outliers — {col}", fontweight="bold")
fig.suptitle("5 — Outlier Detection", fontweight="bold")
save("chart05_outlier_detection")

 chart05_outlier_detection.png


In [8]:
#  Avg Price per SqFt by State
fig, ax = plt.subplots(figsize=(10,5))
state_avg = df.groupby('State')['Price_per_SqFt'].mean().sort_values(ascending=False)
bars = ax.barh(state_avg.index, state_avg.values,
               color=sns.color_palette("Blues_r", len(state_avg)))
ax.bar_label(bars, fmt='₹%.0f', padding=4, fontsize=9)
ax.set_title("6 — Avg Price per SqFt by State", fontweight="bold")
save("chart06_price_per_sqft_by_state")

 chart06_price_per_sqft_by_state.png


In [9]:
# Avg Price by City (Top 15)
fig, ax = plt.subplots(figsize=(11,5))
city_avg = df.groupby('City')['Price_in_Lakhs'].mean().sort_values(ascending=False).head(15)
bars = ax.bar(city_avg.index, city_avg.values,
              color=sns.color_palette("Greens_r", len(city_avg)))
ax.bar_label(bars, fmt='₹%.1fL', padding=3, fontsize=8)
ax.set_title("7 — Avg Property Price by City (Top 15)", fontweight="bold")
ax.set_xlabel("City"); ax.set_ylabel("Avg Price (Lakhs ₹)")
plt.xticks(rotation=45, ha='right')
save("chart07_avg_price_by_city")

 chart07_avg_price_by_city.png


In [10]:
#  Median Property Age by Locality
fig, ax = plt.subplots(figsize=(11,5))
if 'Age_of_Property' in df.columns:
    loc_age = df.groupby('Locality')['Age_of_Property'].median().sort_values(ascending=False).head(15)
    ax.barh(loc_age.index, loc_age.values, color=sns.color_palette("Oranges_r", len(loc_age)))
    ax.set_title("8 — Median Property Age by Locality (Top 15)", fontweight="bold")
    ax.set_xlabel("Median Age (Years)")
save("chart08_median_age_by_locality")

 chart08_median_age_by_locality.png


In [11]:
# BHK Distribution by City
fig, ax = plt.subplots(figsize=(11,5))
top_cities = df['City'].value_counts().head(6).index
bhk_pivot  = (df[df['City'].isin(top_cities)]
              .groupby(['City','BHK']).size().unstack(fill_value=0))
bhk_pivot.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='white')
ax.set_title("9 — BHK Distribution by City", fontweight="bold")
ax.set_xlabel("City"); ax.set_ylabel("Count")
ax.legend(title="BHK", bbox_to_anchor=(1.01,1))
plt.xticks(rotation=30, ha='right')
save("chart09_bhk_by_city")

 chart09_bhk_by_city.png


In [12]:
# Price in Top 5 Localities
fig, ax = plt.subplots(figsize=(10,5))
top_loc  = df.groupby('Locality')['Price_in_Lakhs'].mean().sort_values(ascending=False).head(5).index
loc_data = df[df['Locality'].isin(top_loc)]
sns.stripplot(data=loc_data, x='Locality', y='Price_in_Lakhs',
              jitter=True, alpha=0.5, palette="Set1", ax=ax, size=5)
sns.pointplot(data=loc_data, x='Locality', y='Price_in_Lakhs',
              estimator=np.median, color='black', ax=ax, markers='D', linestyles='--')
ax.set_title("10 — Price in Top 5 Expensive Localities", fontweight="bold")
plt.xticks(rotation=20, ha='right')
save("chart10_price_trends_top_localities")

 chart10_price_trends_top_localities.png


In [13]:
# Correlation Heatmap 
fig, ax = plt.subplots(figsize=(12,9))
key_cols = [c for c in df.select_dtypes(include=np.number).columns
            if c not in ['ID','Year_Built','Appreciation_Rate']][:16]
corr = df[key_cols].corr()
sns.heatmap(corr, mask=np.triu(np.ones_like(corr,dtype=bool)),
            annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            linewidths=0.4, annot_kws={"size":8}, ax=ax)
ax.set_title("11 — Correlation Heatmap", fontweight="bold")
save("chart11_correlation_heatmap")

 chart11_correlation_heatmap.png


In [14]:
# Nearby Schools vs Price per SqFt 
fig, ax = plt.subplots(figsize=(9,5))
sns.scatterplot(data=df, x='Nearby_Schools', y='Price_per_SqFt',
                hue='Good_Investment', palette={0:'#e74c3c',1:'#27ae60'},
                alpha=0.55, s=30, ax=ax)
z = np.polyfit(df['Nearby_Schools'].dropna(),
               df.loc[df['Nearby_Schools'].notna(),'Price_per_SqFt'], 1)
x_line = np.linspace(df['Nearby_Schools'].min(), df['Nearby_Schools'].max(), 100)
ax.plot(x_line, np.poly1d(z)(x_line), 'k--', linewidth=1.2, label='Trend')
ax.set_title("12 — Nearby Schools vs Price per SqFt", fontweight="bold")
ax.legend(title='Good Investment')
save("chart12_schools_vs_price")

 chart12_schools_vs_price.png


In [15]:
# Nearby Hospitals vs Price
fig, ax = plt.subplots(figsize=(9,5))
hosp_avg = df.groupby('Nearby_Hospitals')['Price_per_SqFt'].mean().reset_index()
ax.bar(hosp_avg['Nearby_Hospitals'], hosp_avg['Price_per_SqFt'],
       color="#8E44AD", alpha=0.75, edgecolor='white')
ax.set_title("13 — Nearby Hospitals vs Avg Price per SqFt", fontweight="bold")
ax.set_xlabel("Nearby Hospitals"); ax.set_ylabel("Avg Price per SqFt (₹)")
save("chart13_hospitals_vs_price")

 chart13_hospitals_vs_price.png


In [16]:
# Price by Furnished Status
fig, ax = plt.subplots(figsize=(8,5))
order = df.groupby('Furnished_Status')['Price_in_Lakhs'].median().sort_values(ascending=False).index
sns.violinplot(data=df, x='Furnished_Status', y='Price_in_Lakhs',
               order=order, palette="Pastel1", inner='quartile', ax=ax)
ax.set_title("14 — Price by Furnished Status", fontweight="bold")
save("chart14_price_by_furnished_status")

 chart14_price_by_furnished_status.png


In [17]:
# Price per SqFt by Facing
fig, ax = plt.subplots(figsize=(9,5))
facing_data = df.groupby('Facing')['Price_per_SqFt'].agg(['mean','std']).reset_index()
ax.bar(facing_data['Facing'], facing_data['mean'],
       yerr=facing_data['std'], capsize=5,
       color=sns.color_palette("Set3", len(facing_data)), edgecolor='white')
ax.set_title("15 — Price per SqFt by Facing Direction", fontweight="bold")
ax.set_xlabel("Facing Direction"); ax.set_ylabel("Avg Price per SqFt (₹)")
save("chart15_price_by_facing")

 chart15_price_by_facing.png


In [18]:
# Owner Type Pie Chart 
fig, ax = plt.subplots(figsize=(7,5))
otype = df['Owner_Type'].value_counts()
ax.pie(otype.values, labels=otype.index, autopct='%1.1f%%',
       colors=sns.color_palette("Set2", len(otype)),
       wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax.set_title("16 — Properties by Owner Type", fontweight="bold")
save("chart16_owner_type")

 chart16_owner_type.png


In [19]:
# Availability Status
fig, ax = plt.subplots(figsize=(8,5))
status = df['Availability_Status'].value_counts()
bars   = ax.barh(status.index, status.values,
                 color=['#27ae60','#f39c12','#e74c3c'], edgecolor='white')
ax.bar_label(bars, padding=4)
ax.set_title("17 — Availability Status", fontweight="bold")
save("chart17_availability_status")

 chart17_availability_status.png


In [20]:
# Parking Space vs Price
fig, ax = plt.subplots(figsize=(9,5))
park_avg = df.groupby('Parking_Space')['Price_in_Lakhs'].mean().reset_index()
bars = ax.bar(park_avg['Parking_Space'].astype(str), park_avg['Price_in_Lakhs'],
              color=sns.color_palette("Blues", len(park_avg)), edgecolor='white')
ax.bar_label(bars, fmt='₹%.1fL', padding=3, fontsize=9)
ax.set_title("18 — Parking Space vs Avg Price", fontweight="bold")
ax.set_xlabel("Parking Spaces"); ax.set_ylabel("Avg Price (Lakhs ₹)")
save("chart18_parking_vs_price")

 chart18_parking_vs_price.png


In [21]:
# Amenity Score vs Price per SqFt
fig, ax = plt.subplots(figsize=(9,5))
amenity_avg = df.groupby('Amenity_Score')['Price_per_SqFt'].mean().reset_index()
ax.plot(amenity_avg['Amenity_Score'], amenity_avg['Price_per_SqFt'],
        marker='o', color='#e67e22', linewidth=2, markersize=8)
ax.fill_between(amenity_avg['Amenity_Score'], amenity_avg['Price_per_SqFt'],
                alpha=0.15, color='#e67e22')
ax.set_title("19 — Amenity Score vs Price per SqFt", fontweight="bold")
ax.set_xlabel("Amenity Count"); ax.set_ylabel("Avg Price per SqFt (₹)")
save("chart19_amenities_vs_price")

 chart19_amenities_vs_price.png


In [22]:
# Public Transport vs Investment 
fig, ax = plt.subplots(figsize=(9,5))
t_inv = (df.groupby('Public_Transport_Accessibility')['Good_Investment']
           .value_counts(normalize=True).mul(100)
           .rename('Pct').reset_index()
           .pivot(index='Public_Transport_Accessibility',
                  columns='Good_Investment', values='Pct')
           .fillna(0)
           .reindex(['Low','Medium','High']))
t_inv.columns = ['Not Good','Good Investment']
t_inv.plot(kind='bar', ax=ax, color=['#e74c3c','#27ae60'], edgecolor='white', width=0.6)
ax.set_title("20 — Public Transport vs Investment Potential", fontweight="bold")
ax.set_xlabel("Transport Accessibility"); ax.set_ylabel("Percentage (%)")
plt.xticks(rotation=0)
save("chart20_transport_vs_investment")
 
print("\n✅ Step 2 Done! All 20 charts saved in eda_charts/")
 

 chart20_transport_vs_investment.png

✅ Step 2 Done! All 20 charts saved in eda_charts/
